# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records and Demand Dataset:

In this notebook, we will aggregrated HVFHV dataset hourly and merge it with hourly demand dataset.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
from pyspark.sql.functions import col, avg, count
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_demand+hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"

Full HVFHV dataset:

In [ ]:
full_hvfhv_sdf_dir = base_dir + '/developed/merged_data/full_hvfhv'
full_hvfhv_sdf = spark.read.parquet(full_hvfhv_sdf_dir)
full_hvfhv_sdf.show(5)

# Create the Utilization Rate by HVFHV License Number Dataset:

According to TLC driver income rules, the utilization rate is the percentage of time that high-volume drivers are transporting passengers. It is calculated by dividing the time spent with a passenger by the total time that drivers are logged into the app, including time waiting for a dispatch, time en route to pick up a passenger, and time with a passenger.

Here we can only calculate the rough utilization rate since we do not have the information about the time spent waiting for a dispatch in each order. So it is calculated by `trip_time`/(`request_to_pickup_minutes`+`trip_time`).

In [ ]:
full_hvfhv_sdf = full_hvfhv_sdf.filter(full_hvfhv_sdf.request_to_pickup_minutes >= 0)

In [ ]:
# Add new column `utilization_rate`
full_hvfhv_sdf = full_hvfhv_sdf.withColumn(
    'utilization_rate',
    col('trip_time') / (col('request_to_pickup_minutes') + col('trip_time'))
)

full_hvfhv_sdf.show(5)

In [ ]:
full_hvfhv_sdf = full_hvfhv_sdf.filter((full_hvfhv_sdf.utilization_rate >= 0) &
                                       (full_hvfhv_sdf.utilization_rate <= 1))

In [ ]:
num_rows = full_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = full_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
# Group by `pickup_hour` and `hvfhs_license_num`
# then calculate the average of `utilization_rate`
utilization_rate_sdf = full_hvfhv_sdf.groupBy(
    "pickup_hour", 
    "hvfhs_license_num"
).agg(
    avg("utilization_rate").alias("avg_utilization_rate")
).orderBy(
    "pickup_hour", 
    "hvfhs_license_num"
)

utilization_rate_sdf.show(5)

# Aggregated Hourly Demand Dataset:

In [ ]:
# Group by `pickup_hour`, `pickup_date``, `day_type`,
# then count the number of records for each group
hourly_demand_sdf = full_hvfhv_sdf.groupBy("pickup_hour", "pickup_date", "PULocationID") \
                                    .count() \
                                    .withColumnRenamed("count", "hourly_demand")\
                                    .withColumnRenamed("pickup_hour", "hour")\
                                    .withColumnRenamed("pickup_date", "date")\
                                    .withColumnRenamed("PULocationID", "location")
hourly_demand_sdf.show(5)

# Aggregated Full HVFHV Dataset by Hour:

In [ ]:
COLS = ["trip_miles", "trip_time", "utilization_rate", "base_passenger_fare", 
        "total_fare_amount", "tolls", "bcf", "sales_tax", "congestion_surcharge", 
        "airport_fee", "tips", "driver_pay", "shared_request_flag", 
        "shared_match_flag", "wav_request_flag", "wav_match_flag", 
        "request_to_pickup_minutes", "trip_speed", "total_revenue"]

# Drop unused columns
full_hvfhv_sdf = full_hvfhv_sdf.drop('hvfhs_license_num', 'DOLocationID', 
                                     'dropoff_date', 'dropoff_hour', )

# Create a list of aggregation expressions
agg_exprs = [F.avg(col).alias(f'avg_{col}') for col in COLS]

# Perform the aggregation
hourly_full_hvfhv_sdf = full_hvfhv_sdf.groupBy('pickup_date', 'day_of_week', 'month', 'pickup_hour', 'PULocationID').agg(*agg_exprs)

hourly_full_hvfhv_sdf.show(5)

In [ ]:
# Add new column `day_type` to indicate it is weekday (0) or weekend (1)
hourly_full_hvfhv_sdf = hourly_full_hvfhv_sdf.withColumn(
    "day_type",
    F.when(hourly_full_hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", 
                                                      "Thursday", "Friday"]), 0)
    .otherwise(1)
)

# Drop unused columns
hourly_full_hvfhv_sdf = hourly_full_hvfhv_sdf.drop('day_of_week')

hourly_full_hvfhv_sdf.show(5)

In [ ]:
num_rows = hourly_full_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_full_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hourly_full_hvfhv_sdf.describe().show()

# Create the Hourly Full HVFHV and Pickup Demand Related Dataset:

In [ ]:
# Merge `hourly_demand_sdf` and `hourly_full_hvfhv_sdf` on hour and date
hourly_demand_hvfhv_sdf = hourly_demand_sdf.join(hourly_full_hvfhv_sdf, 
                                                 (hourly_full_hvfhv_sdf.pickup_hour == hourly_demand_sdf.hour) &
                                                 (hourly_full_hvfhv_sdf.pickup_date == hourly_demand_sdf.date) &
                                                 (hourly_full_hvfhv_sdf.PULocationID == hourly_demand_sdf.location), 
                                                 how='left').drop(hourly_demand_sdf['date']).drop(hourly_demand_sdf['location'])

hourly_demand_hvfhv_sdf = hourly_demand_hvfhv_sdf.drop(hourly_demand_sdf['hour'])
hourly_demand_hvfhv_sdf.show(5)

In [ ]:
num_rows = hourly_demand_hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_demand_hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hourly_demand_hvfhv_sdf.printSchema()

# Save the Merged Datasets:

Utilization rate dataset:

In [ ]:
utilization_rate_sdf_dir = base_dir + '/developed/merged_data'
file_name = 'utilization_rate'
utilization_rate_sdf_path = os.path.join(utilization_rate_sdf_dir, file_name)
utilization_rate_sdf.write.mode('overwrite').parquet(utilization_rate_sdf_path)

Hourly demand HVFHV dataset:

In [ ]:
hourly_demand_hvfhv_sdf_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_hvfhv'
hourly_demand_hvfhv_sdf_path = os.path.join(hourly_demand_hvfhv_sdf_dir, file_name)
hourly_demand_hvfhv_sdf.write.mode('overwrite').parquet(hourly_demand_hvfhv_sdf_path)